In [53]:
"""
APSP: All-Pairs Shortest Paths
"""

'\nAPSP: All-Pairs Shortest Paths\n'

In [45]:
import math

def extend_shortest_paths(L_prev, W, n):
    L_new = [[math.inf] * n for _ in range(n)]
    
    for i in range(n):
        for j in range(n):
            for k in range(n):
                L_new[i][j] = min(L_new[i][j],
                                  L_prev[i][k] + W[k][j])
    return L_new

In [46]:
def slow_apsp(W):
    n = len(W)
    
    # L^(1) = W
    L = W
    
    for _ in range(n - 2):
        L = extend_shortest_paths(L, W, n)
    
    return L

In [47]:
def slow_apsp_with_layers(W):
    n = len(W)
    L = [None] * n
    L[0] = W

    for r in range(1, n):
        L[r] = extend_shortest_paths(L[r-1], W, n)

    return L

In [48]:
def faster_apsp(W):
    n = len(W)
    
    L = W
    m = 1
    
    while m < n - 1:
        L = extend_shortest_paths(L, L, n)  # square it
        m *= 2
    
    return L

In [49]:
def faster_apsp_with_layers(W):
    n = len(W)
    layers = {1: W}
    
    r = 1
    while r < n - 1:
        layers[2*r] = extend_shortest_paths(layers[r], layers[r], n)
        r *= 2
    
    return layers

In [50]:
"""
Slow APSP:
Let's try paths with 1 more edge each time.

Faster APSP:
Let's double how far we can go each time.

Any shortest path has at most n−1 edges, so:

Slow: takes n−1 iterations
Faster: reaches ≥ n−1 using doubling

| Method | Progression  | # Steps  |
| -------| ------------ | -------- |
| Slow   | L¹ → L² → L³ | O(n)     |
| Faster | L¹ → L² → L⁴ | O(log n) |

GRAPH:

0 --1--> 1 --2--> 2
 \--------------5--->

This means:
0→1 costs 1
1→2 costs 2
0→2 costs 5 (direct edge, but not optimal)

INITIAL MATRIX W (L^(1)):
[0   1   5]
[∞   0   2]
[∞   ∞   0]


STEP: W ⊗ W (this is where improvement happens)

We try all "middle nodes k" for each pair (i, j).

Focus on entry (0, 2):

Try k = 0:
0→0 + 0→2 = 0 + 5 = 5

Try k = 1:
0→1 + 1→2 = 1 + 2 = 3 BEST PATH FOUND

Try k = 2:
0→2 + 2→2 = 5 + 0 = 5


RESULTING MATRIX (L^(2)):
[0   1   3]
[∞   0   2]
[∞   ∞   0]


WHAT JUST HAPPENED (INTUITION):
The matrix multiplication is not “multiplying numbers”.
It is testing ALL possible middle stops k.

So instead of only using the direct edge 0→2 = 5,
it discovers a better two-step route:
0 → 1 → 2 = 1 + 2 = 3

This is why the value changes from 5 → 3.

---

WHY THIS ENABLES FAST APSP:

Slow version:
- builds paths like a staircase
- 1 edge → 2 edges → 3 edges → 4 edges

Fast version:
- combines whole blocks of paths
- 1 edge → 2 edges → 4 edges → 8 edges

Because each multiplication already considers ALL middle nodes,
it “jumps” path length instead of stepping one edge at a time.
"""

'\nSlow APSP:\nLet\'s try paths with 1 more edge each time.\n\nFaster APSP:\nLet\'s double how far we can go each time.\n\nAny shortest path has at most n−1 edges, so:\n\nSlow: takes n−1 iterations\nFaster: reaches ≥ n−1 using doubling\n\n| Method | Progression  | # Steps  |\n| -------| ------------ | -------- |\n| Slow   | L¹ → L² → L³ | O(n)     |\n| Faster | L¹ → L² → L⁴ | O(log n) |\n\nGRAPH:\n\n0 --1--> 1 --2--> 2\n \\--------------5--->\n\nThis means:\n0→1 costs 1\n1→2 costs 2\n0→2 costs 5 (direct edge, but not optimal)\n\nINITIAL MATRIX W (L^(1)):\n[0   1   5]\n[∞   0   2]\n[∞   ∞   0]\n\n\nSTEP: W ⊗ W (this is where improvement happens)\n\nWe try all "middle nodes k" for each pair (i, j).\n\nFocus on entry (0, 2):\n\nTry k = 0:\n0→0 + 0→2 = 0 + 5 = 5\n\nTry k = 1:\n0→1 + 1→2 = 1 + 2 = 3   ← BEST PATH FOUND\n\nTry k = 2:\n0→2 + 2→2 = 5 + 0 = 5\n\n\nRESULTING MATRIX (L^(2)):\n[0   1   3]\n[∞   0   2]\n[∞   ∞   0]\n\n\nWHAT JUST HAPPENED (INTUITION):\n\nThe matrix multiplication i

In [52]:
"""
Semiring
switching to the algebra that matches the problem's structure

Standard algebra = how things combine numerically in physics / linear systems
Min-plus algebra = how optimal decisions combine in path problems

Same shape (matrices, multiplication), different meaning underneath.

⊗ behaves like matrix multiplication structurally, but it is a different algebra:
min-plus (tropical) matrix multiplication

normal linear algebra → (+,×)
APSP algebra → (min,+)

(A ⊗ B)[i,j] = min[k](A[i,k] + B[k,j])

| Concept         | Standard linear algebra | APSP (min-plus algebra) |
| --------------- | ----------------------- | ----------------------- |
| addition        | (+)                     | min                     |
| multiplication  | (×)                     | (+)                     |
| identity matrix | 1 on diagonal           | 0 on diagonal           |
| “zero element”  | 0                       | ∞                       |
| matrix product  | sum of products         | min of sums             |
|     L^(0)       | not meaningful here     | identity element        |

Matrix multiplication defined by EXTEND-SHORTEST-PATHS is associative because both:

path concatenation (addition) and
choice of best path (minimum over intermediates)

are operations that can be regrouped without changing the set of all candidate paths.
"""

'\n⊗ behaves like matrix multiplication structurally, but it is a different algebra:\n\nnormal linear algebra → (+,×)\nAPSP algebra → (min,+)\n\n(A ⊗ B)[i,j] = min[k](A[i,k] + B[k,j])\n'

In [54]:
# EXTEND-SHORTEST-PATHS (min-plus) associativity proof

# Define operation:
# (A ⊗ B)[i,j] = min_k (A[i,k] + B[k,j])

# Goal:
# (A ⊗ B) ⊗ C == A ⊗ (B ⊗ C)

# -------------------------------
# LEFT-HAND SIDE
# -------------------------------

# ((A ⊗ B) ⊗ C)[i,j]

# Step 1: expand outer multiplication
LHS = "min_k ( (A ⊗ B)[i,k] + C[k,j] )"

# Step 2: expand (A ⊗ B)
# (A ⊗ B)[i,k] = min_x (A[i,x] + B[x,k])

LHS_expanded = "min_k ( min_x (A[i,x] + B[x,k]) + C[k,j] )"

# Step 3: flatten nested mins
LHS_flat = "min_{k,x} (A[i,x] + B[x,k] + C[k,j])"


# -------------------------------
# RIGHT-HAND SIDE
# -------------------------------

# (A ⊗ (B ⊗ C))[i,j]

# Step 1: expand outer multiplication
RHS = "min_x ( A[i,x] + (B ⊗ C)[x,j] )"

# Step 2: expand (B ⊗ C)
# (B ⊗ C)[x,j] = min_k (B[x,k] + C[k,j])

RHS_expanded = "min_x ( A[i,x] + min_k (B[x,k] + C[k,j]) )"

# Step 3: flatten nested mins
RHS_flat = "min_{x,k} (A[i,x] + B[x,k] + C[k,j])"


# -------------------------------
# FINAL COMPARISON
# -------------------------------

proof = """
LHS = min_{k,x} (A[i,x] + B[x,k] + C[k,j])
RHS = min_{x,k} (A[i,x] + B[x,k] + C[k,j])

Since minimization over all pairs (x,k) is order-independent:
LHS == RHS
⇒ associativity holds
"""

print(proof)


LHS = min_{k,x} (A[i,x] + B[x,k] + C[k,j])
RHS = min_{x,k} (A[i,x] + B[x,k] + C[k,j])

Since minimization over all pairs (x,k) is order-independent:
LHS == RHS
⇒ associativity holds



In [56]:
def initialize(W):
    n = len(W)
    
    L = [row[:] for row in W]
    Pi = [[None] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j and W[i][j] != float('inf'):
                Pi[i][j] = i

    return L, Pi

In [55]:
def extend_shortest_paths(L_prev, Pi_prev, W):
    n = len(L_prev)
    
    L_new = [[float('inf')] * n for _ in range(n)]
    Pi_new = [[None] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            # Start with previous best (≤ r-1 edges)
            L_new[i][j] = L_prev[i][j]
            Pi_new[i][j] = Pi_prev[i][j]

            # Try extending via k
            for k in range(n):
                if L_prev[i][k] == float('inf') or W[k][j] == float('inf'):
                    continue

                val = L_prev[i][k] + W[k][j]

                if val < L_new[i][j]:
                    L_new[i][j] = val
                    Pi_new[i][j] = k   # predecessor of j is k

    return L_new, Pi_new

In [57]:
def slow_apsp_w_predecessors(W):
    n = len(W)

    L, Pi = initialize(W)

    for r in range(2, n):
        L, Pi = EXTEND_SHORTEST_PATHS(L, Pi, W)

    return L, Pi

In [58]:
"""
After each matrix squaring step, check whether any diagonal entry L[i,i] is negative.
If so, report that the graph contains a negative-weight cycle.
"""
def faster_apsp_negative_cycle(W):
    n = len(W)

    L = [row[:] for row in W]
    m = 1

    while m < n - 1:
        L_new = [[float('inf')] * n for _ in range(n)]

        for i in range(n):
            for j in range(n):
                for k in range(n):
                    val = L[i][k] + L[k][j]
                    if val < L_new[i][j]:
                        L_new[i][j] = val

        L = L_new
        m *= 2

        for i in range(n):
            if L[i][i] < 0:
                return True

    return False

In [59]:
"""
Computing shortest paths with bounded edges using min-plus powers,
and binary search for the smallest r where a negative diagonal appears.

That r is exactly:
the minimum number of edges in any negative-weight cycle.

What is the shortest cycle I can form if I am only allowed paths up to length r?
it doubles each time, A -> C -> D -> A
r = 2 -> r = 4, finds at 4 then zooms into 3.
"""
def min_length_negative_cycle(W):
    n = len(W)

    # precompute powers L^(1), L^(2), L^(4)
    L_powers = []
    L = [row[:] for row in W]
    L_powers.append(L)

    k = 1
    while k < n:
        L_new = [[float('inf')] * n for _ in range(n)]
        for i in range(n):
            for j in range(n):
                for t in range(n):
                    val = L[i][t] + L[t][j]
                    if val < L_new[i][j]:
                        L_new[i][j] = val

        L = L_new
        L_powers.append(L)
        k *= 2

    def has_negative_cycle(L):
        return any(L[i][i] < 0 for i in range(n))

    result = float('inf')
    L_current = [[float('inf')] * n for _ in range(n)]

    for i in range(n):
        L_current[i][i] = 0

    r = 0

    for i in reversed(range(len(L_powers))):
        candidate = [[float('inf')] * n for _ in range(n)]
        for a in range(n):
            for b in range(n):
                for c in range(n):
                    val = L_current[a][c] + L_powers[i][c][b]
                    if val < candidate[a][b]:
                        candidate[a][b] = val

        if not has_negative_cycle(candidate):
            L_current = candidate
            r += (1 << i)

    return r + 1

In [60]:
"""
Book defines a new nxn matrix in the first loop
but we edit the new matrix in place so its not needed.
"""
def floyd_warshall(W):
    """
    W: adjacency matrix (n x n)
       W[i][j] = weight of edge i→j
       use float('inf') if no edge
       W[i][i] should be 0

    returns:
       D = shortest-path distance matrix
    """
    n = len(W)

    # Copy input matrix so we don't overwrite original
    D = [row[:] for row in W]

    # Main triple loop
    for k in range(n):          # intermediate vertex
        for i in range(n):      # source
            for j in range(n):  # destination
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]

    return D

In [61]:
inf = float('inf')

W = [
    [0,   3,   inf, 7],
    [8,   0,   2,   inf],
    [5,   inf, 0,   1],
    [2,   inf, inf, 0]
]

D = floyd_warshall(W)

for row in D:
    print(row)

[0, 3, 5, 6]
[5, 0, 2, 3]
[3, 6, 0, 1]
[2, 5, 7, 0]


In [62]:
def floyd_warshall_with_path(W):
    n = len(W)
    D = [row[:] for row in W]
    Pi = [[None]*n for _ in range(n)]

    # predecessors
    for i in range(n):
        for j in range(n):
            if i != j and W[i][j] != float('inf'):
                Pi[i][j] = i

    for k in range(n):
        for i in range(n):
            for j in range(n):
                if D[i][k] + D[k][j] < D[i][j]:
                    D[i][j] = D[i][k] + D[k][j]
                    Pi[i][j] = Pi[k][j]

    return D, Pi

In [63]:
inf = float('inf')

W = [
    [0,   3,   inf, 7],
    [8,   0,   2,   inf],
    [5,   inf, 0,   1],
    [2,   inf, inf, 0]
]

D = floyd_warshall_with_path(W)

for row in D:
    print(row)

[[0, 3, 5, 6], [5, 0, 2, 3], [3, 6, 0, 1], [2, 5, 7, 0]]
[[None, 0, 1, 2], [3, None, 1, 2], [3, 0, None, 2], [3, 0, 1, None]]
